In [0]:
# ============================================================
# Silver — Source 17: GA4 Analytics Export
#
# Transformations:
#   - Cast event_date YYYYMMDD string to date
#   - Cast event_timestamp microseconds to timestamp
#   - Extract transaction_id (order_id) from event_params for purchase events
#   - Extract product_sku from event_params for item events
#   - Normalise platform, device fields
#   - user_id null is valid (anonymous)
#   - Deduplicate on user_pseudo_id + event_timestamp
#
# Source:  bronze.src_17_analytics.events
# Target:  silver.src_17_analytics.events
# Quarantine: silver.quarantine.src_17_analytics
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
BRONZE_CATALOG = 'bronze'
SILVER_CATALOG = 'silver'
TARGET_TABLE = f'{SILVER_CATALOG}.src_17_analytics.events'
QUARANTINE_TABLE = f'{SILVER_CATALOG}.quarantine.src_17_analytics'
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SILVER_CATALOG}.src_17_analytics')
print('Silver Source 17 GA4 — starting...')


In [0]:
bronze = spark.table(f'{BRONZE_CATALOG}.src_17_analytics.events')
total = bronze.count()
print(f'Bronze rows: {total}')

# Cast event_date YYYYMMDD string to date
df = bronze \
    .withColumn('event_date', F.to_date(F.col('event_date').cast('string'), 'yyyyMMdd')) \
    .withColumn('event_timestamp', (F.col('event_timestamp') / 1e6).cast('timestamp')) \
    .withColumn('platform', F.upper(F.trim(F.col('platform')))) \
    .withColumn('device_category', F.lower(F.trim(F.col('device_category')))) \
    .withColumn('geo_country', F.upper(F.trim(F.col('geo_country'))))

# Extract transaction_id from event_params for purchase events
# event_params is array of structs with key/value
df = df.withColumn('transaction_id',
    F.when(
        F.col('event_name') == 'purchase',
        F.expr("""
            get(filter(event_params, x -> x.key = 'transaction_id'), 0).value.string_value
        """)
    ).otherwise(None)
).withColumn('transaction_id', F.expr('try_cast(transaction_id as long)'))

# Extract item_id (product_sku) from event_params
df = df.withColumn('item_id',
    F.expr("""
        get(filter(event_params, x -> x.key = 'items'), 0).value.string_value
    """)
)

# Bad rows
bad = df.filter(
    F.col('user_pseudo_id').isNull() |
    F.col('event_name').isNull() |
    F.col('event_date').isNull()
).withColumn('quarantine_reason', F.lit('failed_validation')) \
 .withColumn('source_table', F.lit('events'))

# Good rows — dedup on user_pseudo_id + event_timestamp
good = df.filter(
    F.col('user_pseudo_id').isNotNull() &
    F.col('event_name').isNotNull() &
    F.col('event_date').isNotNull()
).dropDuplicates(['user_pseudo_id', 'event_timestamp'])

bad_count = bad.count()
good_count = good.count()
purchase_count = good.filter(F.col('event_name') == 'purchase').count()
print(f'GA4 events: {total} total → {good_count} clean, {bad_count} quarantined')
print(f'Purchase events: {purchase_count}')

if spark.catalog.tableExists(TARGET_TABLE):
    dt = DeltaTable.forName(spark, TARGET_TABLE)
    dt.alias('t').merge(good.alias('s'),
        't.user_pseudo_id = s.user_pseudo_id AND t.event_timestamp = s.event_timestamp') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    good.write.format('delta').mode('overwrite').saveAsTable(TARGET_TABLE)
print('Written')

if bad_count > 0:
    bad.select(
        F.lit('src_17_analytics').alias('source'),
        F.col('source_table'),
        F.col('quarantine_reason'),
        F.current_timestamp().alias('quarantined_at'),
        F.to_json(F.struct(*[c for c in bad.columns if c not in ['quarantine_reason','source_table']])).alias('raw_record')
    ).write.format('delta').mode('append').option('mergeSchema','true').saveAsTable(QUARANTINE_TABLE)


In [0]:
count = spark.sql(f'SELECT COUNT(*) as cnt FROM {TARGET_TABLE}').collect()[0]['cnt']
print(f'silver.src_17_analytics.events: {count} rows')
spark.sql(f'''
    SELECT event_name, COUNT(*) as cnt
    FROM {TARGET_TABLE}
    GROUP BY event_name ORDER BY cnt DESC
''').show()
